# NB03 – Data Analysis

**LSE ID:** 250100007

## Purpose

In NB01 I collected every soda Open Food Facts holds for four countries, and in
NB02 I turned those files into a clean table (`sodas_clean.csv`) with one row per
product per country and a sugar value in grams per 100g. This notebook uses that
table to answer the question: **do soft drinks sold in countries with a sugar tax
contain less sugar than those sold in countries without one?**

Three decisions shape the analysis, and I set them out before the code.

## Decision 1: Which countries count as taxed

The United Kingdom and France are taxed. Germany and Italy are not.

Italy is the case that needs stating clearly, because Italian law does contain a
sugar tax: it was enacted in the 2020 Budget Law and has been postponed
repeatedly, most recently to January 2027, so it has never actually applied. No
Italian manufacturer has yet faced a financial reason to reformulate, which is
what this comparison tests, so Italy belongs with the untaxed group. The sources
for all four countries are in NB01.

## Decision 2: What counts as a diet drink

I treat a product as a diet drink when its recorded sugar is exactly **0 g per
100g**.

This split matters because a sugar tax can lower the average sugar on the shelves
in two quite different ways. It can push manufacturers to **reformulate** existing
drinks so they contain less sugar, or it can push them to **launch and stock more
zero-sugar variants** alongside the originals. Both lower the average, but only
the first means the drinks themselves changed. Reporting one overall average would
blur the two together, so I look at the full-sugar drinks separately.

## Decision 3: Why I report both the mean and the median

The mean is pulled down by every zero-sugar product, while the median barely moves
when a minority of products sit at zero. Reporting both, alongside the number of
products in each group, shows whether a country's low average comes from its
drinks containing less sugar or from its shelves carrying more diet variants.

In [2]:
import pandas as pd
import plotly.express as px

## The data this notebook uses

This notebook does not touch the API. NB01 collected every soda Open Food Facts
holds for the four countries and saved the raw JSON, and NB02 flattened those
files into a table and removed products with no sugar value recorded. NB03 starts
from that table.

The input is `../data/processed/sodas_clean.csv`: **6,086 rows**, one per product
per country, with five columns carried over from NB02 — `country`, `barcode`,
`product_name`, `brand` and `sugar_content_per_100g`.

Two things from NB02 shape how I read the results here, so they are worth
restating rather than leaving in the previous notebook:

- **The country groups are very unequal.** France contributes 3,497 products with
  a sugar value and Italy 323. This reflects how thoroughly volunteers have
  catalogued each market, not anything about the drinks themselves.
- **Data completeness also differs by country.** The United Kingdom was missing
  12.7% of its sugar values against Italy's 4.2%, so the British products in this
  table are a slightly more filtered set than the Italian ones.

Both matter when comparing countries, and I return to them in the conclusions.

In [5]:
df = pd.read_csv("../data/processed/sodas_clean.csv")

# Confirm NB02's output arrived intact: 6,086 rows with a sugar value.
print(df.shape)
print(df.head())
print(len(df))

(6086, 5)
  country        barcode         product_name      brand  \
0   italy  8002516010223          La classica  Tomarchio   
1   italy  3270190005261  PULP' Saveur Orange  Carrefour   
2   italy  4060800129680          Pepsi lemon      Pepsi   
3   italy  4060800001771           Pepsi-cola      pepsi   
4   italy  5449000005090                Fanta      fanta   

   sugar_content_per_100g  
0                    11.0  
1                     8.4  
2                    10.7  
3                    10.9  
4                    11.8  
6086


In [6]:
# Classify each country by tax status and give it a display name for charts.
# Putting the classification in a dictionary keeps the decision visible in the
# code rather than buried in a filter.
TAX_STATUS = {
    "united-kingdom": "Sugar tax",
    "france": "Sugar tax",
    "germany": "No sugar tax",
    "italy": "No sugar tax",
}

COUNTRY_NAMES = {
    "united-kingdom": "United Kingdom",
    "france": "France",
    "germany": "Germany",
    "italy": "Italy",
}

df["tax_status"] = df["country"].map(TAX_STATUS)
df["country_name"] = df["country"].map(COUNTRY_NAMES)

# A diet drink is one with no sugar at all.
df["is_diet"] = df["sugar_content_per_100g"] == 0

df.head()

,country,barcode,product_name,brand,sugar_content_per_100g,tax_status,country_name,is_diet
0,italy,8002516010223,La classica,Tomarchio,11.0,No sugar tax,Italy,False
1,italy,3270190005261,PULP' Saveur Orange,Carrefour,8.4,No sugar tax,Italy,False
2,italy,4060800129680,Pepsi lemon,Pepsi,10.7,No sugar tax,Italy,False
3,italy,4060800001771,Pepsi-cola,pepsi,10.9,No sugar tax,Italy,False
4,italy,5449000005090,Fanta,fanta,11.8,No sugar tax,Italy,False


## Preparing the columns the analysis needs

The table arrives with the facts about each product. The analysis needs three
further columns, all derived from what is already there rather than fetched from
anywhere new.

**`tax_status`** groups the four countries into taxed and untaxed. I define the
grouping in a dictionary rather than inside a filter so that the classification is
visible in the code: a reader can see immediately that Italy sits with the untaxed
countries, and NB01 explains why. Built with `.map()`, which reads each country
name and writes out the matching label.

**`country_name`** turns the file-derived keys into readable labels for tables and
charts, so `united-kingdom` displays as `United Kingdom`.

**`is_diet`** marks products whose recorded sugar is exactly zero. This is the
column the analysis turns on. A sugar tax can lower the average sugar on a
country's shelves in two different ways — by manufacturers reformulating existing
drinks, or by them stocking more zero-sugar variants alongside the originals — and
a single overall average cannot tell those apart. Splitting on this column lets me
look at each separately.

In [ ]:
#Finding 1: the headline comparison, all drinks together.
findings = (
    df.groupby("country_name")["sugar_content_per_100g"]
     .agg(["count", "mean", "median"])
     .round(2)
     )

print(findings)
print()
print(df.groupby("tax_status")["sugar_content_per_100g"].agg(["count", "mean", "median"]).round(2))

                count  mean  median
country_name                       
France           3497  6.90    7.51
Germany          1819  6.49    7.00
Italy             323  7.70    8.80
United Kingdom    447  4.33    3.90

              count  mean  median
tax_status                       
No sugar tax   2142  6.67     7.1
Sugar tax      3944  6.61     7.2


In [ ]:
# Decision 2 draws the diet line at exactly zero. How many products sit just
# above it? If a lot cluster between 0 and 0.5, the cutoff is doing more work
# than it should and I need to say so.
just_above = df[(df["sugar_content_per_100g"] > 0) & (df["sugar_content_per_100g"] <= 0.5)]
print(f"Exactly zero: {df['is_diet'].sum()}")
print(f"Between 0 and 0.5: {len(just_above)}")

Exactly zero: 1022
Between 0 and 0.5: 372


### Is the diet cutoff doing too much work?

Defining a diet drink as exactly 0 g of sugar is a decision, not a fact about the
data, so it is worth checking how sensitive the result is to where I draw the
line. If a large number of products sat just above zero, the cutoff would be
splitting a continuous range at an arbitrary point.

**1,022 products record exactly zero, and a further 372 fall between 0 and 0.5 g
per 100g.** Moving the line to 0.5 would therefore enlarge the diet group by
roughly a third, which is enough that the choice needs justifying rather than
assuming.

Two reasons for keeping the stricter definition. Exactly zero is an unambiguous
value that means the same thing in every country's records, whereas 0.5 is a
threshold I would be importing from labelling conventions rather than from my
data. And the question I am asking is about products with no sugar at all, since
those are the ones a manufacturer launches as an alternative rather than as a
reformulation.

In [10]:
# Finding 2: how much of each country's range is zero-sugar?
diet_share = (
    df.groupby("country_name")["is_diet"]
    .agg(["sum", "mean"])
    .rename(columns={"sum": "diet_products", "mean": "diet_share"})
    .round(3)
)
print(diet_share)

                diet_products  diet_share
country_name                             
France                    562       0.161
Germany                   283       0.156
Italy                      62       0.192
United Kingdom            115       0.257


### Finding 2: how much of each country's range carries no sugar at all

| Country | Zero-sugar products | Share of that country's range |
|---|---|---|
| United Kingdom | 115 of 447 | **25.7%** |
| Italy | 62 of 323 | 19.2% |
| France | 562 of 3,497 | 16.1% |
| Germany | 283 of 1,819 | 15.6% |

A quarter of British sodas contain no sugar at all, well ahead of the other three.
That is consistent with a tax pushing manufacturers to put more zero-sugar
variants on the shelves.

**But tax status does not order this list.** Italy, which has no sugar tax in
force, carries a higher share of zero-sugar drinks than France, which has had one
since 2012. If the tax were the driving force, the two taxed countries should sit
above the two untaxed ones, and they do not. Only the United Kingdom stands apart.

This is the first sign of something the next finding confirms: whatever separates
the United Kingdom from the other three, it is not simply the presence of a sugar
tax, because France has one too and does not behave like it.

In [ ]:
# Finding 3: full-sugar drinks only. If the UK still sits lower here, that points
# to drinks being reformulated. If it rises to meet the others, the UK's low
# average came from its shelves carrying more zero-sugar variants instead.
sugary = df[df["is_diet"] == False]

print(sugary.groupby("country_name")["sugar_content_per_100g"].agg(["count", "mean", "median"]).round(2))

                count  mean  median
country_name                       
France           2935  8.22    8.39
Germany          1536  7.68    7.80
Italy             261  9.53   10.00
United Kingdom    332  5.83    4.60


### Finding 3: the same comparison with zero-sugar drinks removed

Finding 2 raises an obvious objection. If the United Kingdom's average is low
because a quarter of its range sits at zero, then British *drinks* may be no
different from anyone else's — the shelves would simply hold more diet variants.
Removing every zero-sugar product tests that directly. What remains is only the
drinks that do contain sugar.

| Country | Full-sugar products | Mean | Median |
|---|---|---|---|
| Italy | 261 | 9.53 | 10.00 |
| France | 2,935 | 8.22 | 8.39 |
| Germany | 1,536 | 7.68 | 7.80 |
| United Kingdom | 332 | **5.83** | **4.60** |

**The gap does not close.** British full-sugar sodas still contain far less sugar
than everyone else's: a median of 4.60 g per 100g against Germany's 7.80 and
Italy's 10.00, less than half the Italian figure. So the United Kingdom is not
merely stocking more diet drinks alongside unchanged originals. Its sugary drinks
themselves contain less sugar.

Both mechanisms are therefore present in the British data at once — a larger
zero-sugar range *and* less sugar in the drinks that remain — which is what
reformulation in response to a tax would look like.

**France still does not fit.** With diet drinks removed, French sodas average 8.22
against untaxed Germany's 7.68, so the taxed country contains *more* sugar than
the untaxed one. Two taxed countries, opposite results.


In [12]:
# Does moving the diet line from 0 to 0.5 change which countries look diet-heavy?
df["is_diet_05"] = df["sugar_content_per_100g"] <= 0.5
print(df.groupby("country_name")["is_diet_05"].mean().round(3))

country_name
France            0.216
Germany           0.214
Italy             0.229
United Kingdom    0.387
Name: is_diet_05, dtype: float64


### What this comparison can and cannot show

**This is a snapshot, not a before-and-after.** I have what is on the shelves now,
not what was there before each policy. A country could have had lower-sugar drinks
for reasons that have nothing to do with tax — different consumer tastes, earlier
voluntary agreements, a different mix of local brands. My data is consistent with
the UK levy having driven reformulation; it cannot demonstrate that it did.

**The group sizes are very unequal, and that shapes any aggregate.** France
contributes 3,497 products and Italy 323. Averaging across the taxed group makes
France speak for 89% of it, and across the untaxed group makes Germany speak for
85%. That is why the taxed-versus-untaxed comparison in Finding 1 returns almost
nothing — 6.61 against 6.67 — while the country-level view is stark. The
country-by-country comparison is the meaningful one, and I treat the aggregate as
a demonstration of why pooling misleads here rather than as a result.

**Coverage differs by country in ways I cannot correct.** Open Food Facts is
volunteer-built, and NB02 showed the United Kingdom missing 12.7% of its sugar
values against Italy's 4.2%. If the products a British contributor bothers to
complete differ systematically from those an Italian one does, part of the gap I
am measuring comes from that rather than from the drinks.

In [15]:
# Order countries by median sugar so the chart reads from most to least.
COUNTRY_ORDER = ["Italy", "France", "Germany", "United Kingdom"]
TAX_COLOURS = {"Sugar tax": "#3995ba", "No sugar tax": "#c63c4a"}

fig_sugar = px.box(
    sugary,
    x="country_name",
    y="sugar_content_per_100g",
    color="tax_status",
    points="all",
    category_orders={"country_name": COUNTRY_ORDER},
    color_discrete_map=TAX_COLOURS,
    labels={
        "country_name": "Country",
        "sugar_content_per_100g": "Sugar (g per 100g)",
        "tax_status": "",
    },
    template="plotly_white",
)

fig_sugar.update_layout(
    title=dict(
        text="British full-sugar sodas contain around half the sugar of Italian ones",
        font=dict(color="#333333", size=20),
        pad=dict(b=30),
        subtitle=dict(
            text="Zero-sugar drinks excluded. Box spans the 25th to 75th percentile, line is the median.  A small number of products above 25 g per 100g are outside the view",
            font=dict(color="#666666", size=15),
        ),
    ),
    width=1000,
    height=600,
    margin=dict(t=130),
    yaxis=dict(title="Sugar (g per 100g)", range=[0, 25])
)

# 5,064 products is a lot of dots, so make them small and semi-transparent.
fig_sugar.update_traces(marker=dict(size=3, opacity=0.3), jitter=0.4)

# fig_sugar.write_image("figures/sugar_by_country.png", width=1000, height=600, scale=1.5)
fig_sugar.show()

In [16]:
# Proportions, so a bar chart is the right tool here.
diet_chart = (
    df.groupby(["country_name", "tax_status"])["is_diet"]
    .mean()
    .reset_index()
)
diet_chart["diet_percent"] = (diet_chart["is_diet"] * 100).round(1)

fig_diet = px.bar(
    diet_chart,
    x="country_name",
    y="diet_percent",
    color="tax_status",
    category_orders={"country_name": ["United Kingdom", "Italy", "France", "Germany"]},
    color_discrete_map=TAX_COLOURS,
    labels={
        "country_name": "Country",
        "diet_percent": "Share of sodas with no sugar (%)",
        "tax_status": "",
    },
    text="diet_percent",
    template="plotly_white",
)

fig_diet.update_layout(
    title=dict(
        text="A quarter of British sodas contain no sugar at all, more than any other country",
        font=dict(color="#333333", size=20),
        pad=dict(b=30),
        subtitle=dict(
            text="Products recording exactly 0 g of sugar per 100g, as a share of each country's range",
            font=dict(color="#666666", size=15),
        ),
    ),
    bargap=0.3,
    width=1000,
    height=550,
    margin=dict(t=130),
)

# fig_diet.write_image("figures/diet_share.png", width=1000, height=550, scale=1.5)
fig_diet.show()

### Choosing the charts

**Why a box plot for the sugar comparison.** My midterm used a bar chart of
averages with error bars, and the feedback pointed out that bars suit counts and
proportions rather than averages, because a single bar hides how the values are
spread. A box plot shows the median, the middle half of the products and the full
range, so it answers a question the summary table cannot: whether the United
Kingdom's low figure comes from its whole range sitting lower or from a handful of
very low products pulling an average down.

`px.box` was not covered in the course, so briefly: the line inside each box is the
median, the box spans the 25th to 75th percentile, and the whiskers reach the
furthest products within one and a half times that range. Setting `points="all"`
draws every individual product alongside the box.

**Why a bar chart for the diet share.** The second chart shows a proportion — what
share of each country's range contains no sugar — and proportions are exactly what
bar charts are for. The same criticism does not apply here.

Both charts colour the countries by tax status, which makes the awkward part of
the result visible rather than buried in a table: France is coloured as taxed and
still sits among the high-sugar countries.